# Fine-Tune XTTS On Shona In Colab

This notebook is designed for free Colab GPU use.

It supports two repo-loading paths:
- clone the repo from GitHub
- upload `cloner.zip` directly from your machine into Colab


## Outline

1. Check GPU runtime
2. Mount Drive for checkpoints (strongly recommended)
3. Load the repo into `/content/cloner`
4. Install dependencies
5. Build the filtered Shona dataset from Hugging Face
6. Segment clips to <= 10 s (REQUIRED - XTTS skips longer clips)
7. Start XTTS fine-tuning
8. Resume from checkpoints later


In [1]:
# Step 1: verify that Colab gave you a GPU runtime.
!nvidia-smi || true

import torch
HAS_CUDA = torch.cuda.is_available()
print('cuda:', HAS_CUDA)
if HAS_CUDA:
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No CUDA GPU runtime is attached.')
    print('In Colab use Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')


Tue Jul 14 12:31:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

cuda: True
device: Tesla T4


In [ ]:
# Step 2: mount Drive so checkpoints and the dataset survive disconnects.
# A 20-30 hour run WILL disconnect at least once; without Drive you lose everything.
USE_DRIVE = True

PROJECT_ROOT = '/content/cloner'
CHECKPOINT_DIR = '/content/checkpoints/shona_xtts'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/shona_xtts'
    CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/shona_xtts'

print('CHECKPOINT_DIR =', CHECKPOINT_DIR)


## Step 3A - Clone From GitHub

Use this to clone the repo into Colab.


In [3]:
# Step 3A: clone from GitHub.
# Replace the URL before running.
GIT_REPO_URL = 'https://github.com/Rakinzi/cloner'

import os, shutil
if GIT_REPO_URL:
    if os.path.exists(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)
    !git clone "$GIT_REPO_URL" "$PROJECT_ROOT"
    !ls -la "$PROJECT_ROOT"
else:
    print('Set GIT_REPO_URL before running this cell.')


Cloning into '/content/cloner'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 52 (delta 15), reused 48 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 238.43 KiB | 3.18 MiB/s, done.
Resolving deltas: 100% (15/15), done.
total 684
drwxr-xr-x 6 root root   4096 Jul 14 12:31 .
drwxr-xr-x 1 root root   4096 Jul 14 12:31 ..
drwxr-xr-x 5 root root   4096 Jul 14 12:31 app
-rw-r--r-- 1 root root      0 Jul 14 12:31 app.py
-rw-r--r-- 1 root root    287 Jul 14 12:31 docker-compose.yml
-rw-r--r-- 1 root root    406 Jul 14 12:31 Dockerfile
drwxr-xr-x 8 root root   4096 Jul 14 12:31 .git
-rw-r--r-- 1 root root    607 Jul 14 12:31 .gitignore
drwxr-xr-x 3 root root   4096 Jul 14 12:31 output
-rw-r--r-- 1 root root    694 Jul 14 12:31 pyproject.toml
-rw-r--r-- 1 root root    163 Jul 14 12:31 requirements.txt
drwxr-xr-x 2 root root   4096 Jul 14 12:31 scripts
-rw-r--r-- 1 root roo

In [4]:
# Step 3C: verify the repo is present before continuing.
import os
assert os.path.exists(PROJECT_ROOT), 'Repo missing. Run Step 3A first.'
assert os.path.exists(f'{PROJECT_ROOT}/pyproject.toml'), 'pyproject.toml missing. The repo did not extract correctly.'
print('Repo is ready at', PROJECT_ROOT)


Repo is ready at /content/cloner


In [5]:
# Step 4: install dependencies.
import os
assert os.path.exists(f'{PROJECT_ROOT}/pyproject.toml'), 'Repo missing. Run Step 3A first.'
%cd /content/cloner
!python -V
!pip install -U pip setuptools wheel
!pip install uv
!pip install -U torchcodec
!uv sync


/content/cloner
Python 3.12.13


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.4 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 51.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 20.8 MB/s  0:00:00
  Attempting uninstall: torchcodec
    Found existing installation: torchcodec 0.11.0+cu128
    Uninstalling torchcodec-0.11.0+cu128:
      Successfully uninstalled torchcodec-0.11.0+cu128
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 135 packages in 3.26s                                       
Prepared 126 packages in 1m 10s                                          
Installed 126 packages in 1.10s                             
 + absl-py==2.4.0
 + aiofiles==25.1.0
 + aiohappyeyeballs==2.6.2
 + aiohttp==3.14.1
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyascii==0.3.3
 + anyio==4.13.0
 + attrs==26.1.0
 + audioread==3.1.0
 + certifi==2026.5.20
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.4.1
 + contourpy==1.3.3
 + coqpit-config==0.2.5
 + coqui-tts==0.27.5


## Step 5 — Build and locate the filtered dataset

The next three cells show exactly where the source data comes from, where the generated WAV files will be written, and what was created. If Google Drive is mounted, the output persists there; otherwise it is temporary under `/content`.

In [6]:
# Step 5A: show the source and destination before downloading anything.
import os, shutil, time
assert os.path.exists(f'{PROJECT_ROOT}/scripts/build_hf_xtts_dataset.py'), 'Missing build_hf_xtts_dataset.py. Repo not loaded correctly.'

HF_DATASET = 'manassehzw/sna-dataset-annotated'
HF_DATASET_URL = f'https://huggingface.co/datasets/{HF_DATASET}'
TARGET_DATASET_DIR = f'{DRIVE_ROOT}/sna_xtts_ft_filtered' if 'DRIVE_ROOT' in globals() else '/content/sna_xtts_ft_filtered'

print('SOURCE DATASET')
print('  Hugging Face:', HF_DATASET_URL)
print('\nGENERATED DATASET DESTINATION')
print('  Folder:', TARGET_DATASET_DIR)
print('  WAV files:', f'{TARGET_DATASET_DIR}/wavs')
print('  Metadata:', f'{TARGET_DATASET_DIR}/metadata.csv')
print('  Report:', f'{TARGET_DATASET_DIR}/dataset_report.json')
print('  Storage:', 'Google Drive (persistent)' if TARGET_DATASET_DIR.startswith('/content/drive/') else 'Colab runtime (temporary)')
print('\nFREE SPACE')
!df -h /content | tail -1
if TARGET_DATASET_DIR.startswith('/content/drive/'):
    print('\nOpen Google Drive:', 'https://drive.google.com/drive/my-drive')


SOURCE DATASET
  Hugging Face: https://huggingface.co/datasets/manassehzw/sna-dataset-annotated

GENERATED DATASET DESTINATION
  Folder: /content/sna_xtts_ft_filtered
  WAV files: /content/sna_xtts_ft_filtered/wavs
  Metadata: /content/sna_xtts_ft_filtered/metadata.csv
  Report: /content/sna_xtts_ft_filtered/dataset_report.json
  Storage: Colab runtime (temporary)

FREE SPACE
overlay         113G   53G   60G  47% /


In [ ]:
# Step 5B: locate or build the dataset. Priority:
#   1. already-built dataset folder (from a previous session)
#   2. pre-built zip uploaded to Drive (fastest: ~2.4 GB vs 27 GB HF download)
#   3. build from Hugging Face (first run only, ~30-60 min)
import os, time, subprocess

DRIVE_ZIP = f'{DRIVE_ROOT}/sna_xtts_ft_v2.zip' if 'DRIVE_ROOT' in globals() else ''

if os.path.exists(TARGET_DATASET_DIR):
    print('Existing dataset found:', TARGET_DATASET_DIR)
    print('Skipping rebuild; run Step 5C next.')
elif DRIVE_ZIP and os.path.exists(DRIVE_ZIP):
    print('Found pre-built dataset zip on Drive:', DRIVE_ZIP)
    TARGET_DATASET_DIR = '/content/sna_xtts_ft_v2'
    if not os.path.exists(TARGET_DATASET_DIR):
        !unzip -q "$DRIVE_ZIP" -d /content/
    print('Dataset ready at', TARGET_DATASET_DIR)
else:
    print('No existing dataset or zip. Building from Hugging Face at:', TARGET_DATASET_DIR, flush=True)
    build_started = time.time()
    subprocess.run(
        ['uv', 'run', 'python', 'scripts/build_hf_xtts_dataset.py',
         '--output', TARGET_DATASET_DIR,
         '--min-quality', '70',
         '--min-duration', '3',
         '--max-duration', '20',
         '--max-clips-per-speaker', '200'],
        cwd='/content/cloner', check=True,
    )
    print(f'Build finished after {(time.time() - build_started)/60:.1f} min')


In [ ]:
# Step 5C: verify and list exactly what was generated.
from pathlib import Path
import json

dataset_dir = Path(TARGET_DATASET_DIR)
assert dataset_dir.exists(), f'Output folder was not created: {dataset_dir}'
wav_count = sum(1 for _ in (dataset_dir / 'wavs').glob('*.wav'))
print('OUTPUT FOLDER:', dataset_dir)
print('WAV FILES:', wav_count)
print('CONTENTS:')
for item in sorted(dataset_dir.iterdir()):
    print(' ', item)
print('\nDISK USAGE:')
!du -sh "$TARGET_DATASET_DIR"
report_path = dataset_dir / 'dataset_report.json'
if report_path.exists():
    print('\nDATASET REPORT:')
    print(json.dumps(json.loads(report_path.read_text()), indent=2))
if str(dataset_dir).startswith('/content/drive/'):
    print('\nPersistent copy is in Google Drive:', 'https://drive.google.com/drive/my-drive')
else:
    print('\nWARNING: This is temporary Colab storage. Copy it to Drive before disconnecting.')


## Step 6 - Segment clips to fit XTTS limits

XTTS training silently skips any clip longer than ~11.6 s and any text over 200
characters. **97% of our clips are longer than that**, so without this step the
trainer would see ~90 clips instead of ~3,250. This step uses Meta's MMS forced
aligner to find word timestamps and cuts each clip at sentence boundaries into
<= 10 s pieces. Takes roughly 15-30 min on a T4, and the result is cached on
Drive so it only runs once.

In [ ]:
# Step 6: segment the dataset (skipped automatically if already done).
# Output goes to Drive when mounted, so this runs only once across sessions.
import os, json, subprocess

if 'DRIVE_ROOT' in globals():
    SEGMENTED_DATASET_DIR = f'{DRIVE_ROOT}/sna_xtts_seg10s'
else:
    SEGMENTED_DATASET_DIR = '/content/sna_xtts_seg10s'

report_path = f'{SEGMENTED_DATASET_DIR}/segment_report.json'
if os.path.exists(report_path):
    print('Segmented dataset already exists:', SEGMENTED_DATASET_DIR)
    print(json.dumps(json.load(open(report_path)), indent=2))
else:
    subprocess.run(
        ['uv', 'run', 'python', 'scripts/segment_dataset.py',
         '--dataset', TARGET_DATASET_DIR,
         '--output', SEGMENTED_DATASET_DIR],
        cwd='/content/cloner', check=True,
    )
    print(json.dumps(json.load(open(report_path)), indent=2))

# Sanity check: training needs thousands of clips, not dozens.
report = json.load(open(report_path))
assert report['output_clips'] > 2000, (
    'Segmentation produced too few clips - check the log above before spending GPU time on training.')
print('\nOK:', report['output_clips'], 'clips /', report['output_hours'], 'hours ready for training.')
print('Tip: once this succeeds, you can delete sna_xtts_ft_v2.zip from Drive to free 2.4 GB.')


In [ ]:
# Step 7: launch XTTS fine-tuning.
# The dataset is Shona. `--xtts-language en` is only the XTTS-supported language token workaround.
# Expect very roughly 1.5-2.5 h per epoch on a T4. Watch the first `print_step`
# logs to get the real it/s, then decide how many epochs your credits allow.
if not HAS_CUDA:
    raise RuntimeError('Stop here until Colab gives you a GPU runtime. Rerun Step 1 after changing runtime type.')

assert os.path.exists(SEGMENTED_DATASET_DIR), f'Segmented dataset not found: {SEGMENTED_DATASET_DIR}. Run Step 6 first.'
%cd /content/cloner
!MPLBACKEND=Agg uv run python scripts/finetune_xtts.py \
  --dataset "$SEGMENTED_DATASET_DIR" \
  --output "$CHECKPOINT_DIR" \
  --xtts-language en \
  --batch-size 1 \
  --grad-accum 84 \
  --epochs 10 \
  --save-step 1000 \
  --workers 2


In [ ]:
# Step 8: resume later if Colab disconnects.
# Set LATEST_CHECKPOINT to the newest run folder under CHECKPOINT_DIR, e.g.
#   /content/drive/MyDrive/shona_xtts/checkpoints/shona_xtts/run-July-23-2026_.../checkpoint_5000.pth
LATEST_CHECKPOINT = ''

if LATEST_CHECKPOINT:
    assert os.path.exists(SEGMENTED_DATASET_DIR), f'Segmented dataset not found: {SEGMENTED_DATASET_DIR}'
    %cd /content/cloner
    !MPLBACKEND=Agg uv run python scripts/finetune_xtts.py \
      --dataset "$SEGMENTED_DATASET_DIR" \
      --output "$CHECKPOINT_DIR" \
      --xtts-language en \
      --resume "$LATEST_CHECKPOINT" \
      --batch-size 1 \
      --grad-accum 84 \
      --epochs 10 \
      --save-step 1000 \
      --workers 2
else:
    print('Set LATEST_CHECKPOINT to resume from an interrupted run.')


## Notes

- **GPU choice:** use a T4. At batch size 1 an A100 burns ~6x the compute units for far less than 6x the speed.
- **Budget:** $10 = 100 compute units = roughly 50-60 T4 hours. A 10-epoch run is
  estimated at 20-35 T4 hours, but verify against the real it/s from the training log.
- **Disk:** each checkpoint is ~5 GB and up to 3 are kept - make sure Drive has ~15 GB free.
- If the Colab session resets, rerun Steps 1-4, then Step 8 with LATEST_CHECKPOINT set.
  Steps 5-6 are skipped automatically because the data is already on Drive.
- XTTS still uses `en` as the internal language token even though the dataset is Shona.
